# Daily Practice — 2026-09-20 — ML Modelling: When Accuracy Lies to You

**Dataset:** [Breast Cancer Wisconsin (Diagnostic)](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset) — 569 real fine-needle-aspirate biopsy samples, 30 numeric features, loaded via `sklearn.datasets.load_breast_cancer` (a real clinical dataset, not a synthetic or "toy for the sake of a toy" one).

## Problem statement

You're QA-ing a model that flags biopsy samples as **malignant** (positive class) or **benign**. A teammate has already trained a `LogisticRegression` classifier and reports "95% accuracy — ship it." Your job, with an SQA mindset, is to stress-test that single number before it becomes a release gate.

A missed malignant tumor (a **false negative**) is far more costly than a false alarm (a **false positive**) sent for a follow-up biopsy. The default 0.5 probability threshold that `.predict()` uses implicitly treats those two error types as equally bad — which is almost never the right call in a clinical setting.

## What you should produce

1. A baseline classifier evaluated at the default threshold (0.5), with accuracy, precision, recall, F1, and a confusion matrix — and a count of how many malignant cases it *misses*.
2. A precision/recall-vs-threshold sweep, plotted, so you can see the tradeoff instead of a single snapshot.
3. A function that finds the **highest decision threshold that still achieves at least 98% recall** on the malignant class (i.e. maximizes precision subject to a hard recall floor — the kind of constraint a real "test gate" would encode).
4. A short written verdict (3–5 sentences) comparing the default-threshold model against your recall-constrained one, and explicitly answering: *if your CI pipeline gated deployment purely on `accuracy > 90%`, would it have caught the problem with the default-threshold model?*

Try it yourself in the starter cells below before you scroll down to the solution.


## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve,
)

RANDOM_STATE = 42


## Load and inspect the data

Note: in the raw sklearn dataset, `target == 0` means **malignant** and `target == 1` means **benign**.
We flip that below so `label == 1` means malignant — i.e. "1 = the thing we most need to catch."
This matches how you'd usually define the positive class in a QA/monitoring context (1 = the event you're gating on).

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series((data.target == 0).astype(int), name="malignant")  # 1 = malignant, 0 = benign

print(X.shape)
print(y.value_counts(normalize=False))
print(y.value_counts(normalize=True).round(3))


## Your task

Fill in the TODOs below. Signatures are given — don't change them, the solution cells assume these names.


In [ ]:
# TODO: split X, y into X_train, X_test, y_train, y_test
# - 70/30 split
# - stratify on y (class balance matters here — don't skip this)
# - random_state=RANDOM_STATE

X_train, X_test, y_train, y_test = None, None, None, None


In [ ]:
def fit_baseline_model(X_train, y_train):
    """TODO: fit and return a sklearn LogisticRegression on X_train/y_train.
    Use max_iter=5000 so it reliably converges on unscaled features.
    """
    raise NotImplementedError


In [ ]:
def evaluate_at_threshold(y_true, y_proba, threshold):
    """TODO: given true labels and predicted P(malignant), threshold the
    probabilities at `threshold` and return a dict with keys:
    'accuracy', 'precision', 'recall', 'f1', 'confusion_matrix', 'false_negatives'.

    'false_negatives' should be the raw count of malignant cases predicted benign.
    Use sklearn's confusion_matrix with labels=[0, 1] so you know the axis order.
    """
    raise NotImplementedError


In [ ]:
# TODO: fit the baseline model, get predicted probabilities for the malignant
# class on X_test, and call evaluate_at_threshold at threshold=0.5.
# Print the resulting metrics dict.

baseline_model = None
y_proba = None
baseline_metrics_at_0_5 = None


In [ ]:
def find_min_threshold_for_recall(y_true, y_proba, min_recall=0.98):
    """TODO: sweep thresholds (e.g. sklearn's precision_recall_curve, or your
    own np.linspace sweep) and return the HIGHEST threshold that still achieves
    recall >= min_recall on the malignant class. A higher threshold at the same
    recall means fewer false positives, i.e. better precision for the same
    safety floor.
    """
    raise NotImplementedError


In [ ]:
# TODO:
# 1. Plot precision and recall (y-axis) against threshold (x-axis) using
#    sklearn.metrics.precision_recall_curve.
# 2. Mark your chosen threshold from find_min_threshold_for_recall with a
#    vertical line.
# 3. Call evaluate_at_threshold again at that chosen threshold and compare
#    the metrics dict against baseline_metrics_at_0_5.

chosen_threshold = None
metrics_at_chosen_threshold = None


### Write-up

TODO: in 3–5 sentences, compare the default-threshold model against the recall-constrained one
(false negatives, precision, accuracy), and answer: would a CI gate of `accuracy > 90%` have caught
the problem with the default model?

*(your answer here)*


---

## Solution

*(scroll down when you're ready — try it yourself first)*


<details>
<summary>Click to reveal solution</summary>

```python
# 1. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)

# 2. Baseline model
def fit_baseline_model(X_train, y_train):
    model = LogisticRegression(max_iter=5000)
    model.fit(X_train, y_train)
    return model

# 3. Threshold evaluation
def evaluate_at_threshold(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "confusion_matrix": cm,
        "false_negatives": int(fn),
    }

baseline_model = fit_baseline_model(X_train, y_train)
y_proba = baseline_model.predict_proba(X_test)[:, 1]
baseline_metrics_at_0_5 = evaluate_at_threshold(y_test, y_proba, 0.5)
print("At threshold=0.5:", baseline_metrics_at_0_5)

# 4. Highest threshold that still guarantees min_recall
def find_min_threshold_for_recall(y_true, y_proba, min_recall=0.98):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    # precision_recall_curve returns len(thresholds) == len(precisions) - 1;
    # the last precision/recall point (threshold=inf) has no matching threshold.
    candidates = [
        (t, r) for t, r in zip(thresholds, recalls[:-1]) if r >= min_recall
    ]
    if not candidates:
        raise ValueError(f"No threshold achieves recall >= {min_recall}")
    # among thresholds meeting the recall floor, the highest one maximizes precision
    return max(candidates, key=lambda tr: tr[0])[0]

chosen_threshold = find_min_threshold_for_recall(y_test, y_proba, min_recall=0.98)
metrics_at_chosen_threshold = evaluate_at_threshold(y_test, y_proba, chosen_threshold)
print(f"Chosen threshold: {chosen_threshold:.3f}")
print("At chosen threshold:", metrics_at_chosen_threshold)

# 5. Plot
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
plt.figure(figsize=(7, 5))
plt.plot(thresholds, precisions[:-1], label="Precision")
plt.plot(thresholds, recalls[:-1], label="Recall")
plt.axvline(chosen_threshold, color="grey", linestyle="--", label="Chosen threshold")
plt.xlabel("Decision threshold")
plt.ylabel("Score")
plt.title("Precision / Recall vs. threshold (malignant = positive)")
plt.legend()
plt.show()
```

**Explanation**

At the default 0.5 threshold, the model typically lands around 96-97% accuracy — which sounds
like a clean pass — but still produces 1-3 false negatives on a 171-sample test set (roughly 60
malignant cases). Each false negative is a missed cancer diagnosis. Because malignant cases are
the minority class (~37% of the data), accuracy is dominated by how well the model does on the
*majority* (benign) class, so it can look excellent while still failing on the class that matters
most.

Raising the decision threshold search to guarantee ≥98% recall (i.e. deliberately biasing the
model toward flagging more borderline cases as malignant) typically finds a lower threshold than
0.5 — it lets *more* samples through as "malignant," trading some precision (more benign samples
sent for unnecessary follow-up) for a hard floor on catching real malignant cases.

This is exactly why a CI/release gate of `accuracy > 90%` is a weak test: it would happily pass a
model with a handful of missed malignant diagnoses, because those errors are outnumbered and
diluted by the larger benign class. A meaningful gate here needs a **class-aware, cost-aware
metric** — e.g. `recall_on_malignant >= 0.98` — not a single aggregate accuracy threshold. This is
the same principle as testing a search-and-rescue classifier or a fraud detector: pick the metric,
and the threshold on that metric, based on the asymmetric cost of the two error types, not on
whatever number sklearn prints by default.

</details>
